##  ***Hull-White Model Calibration & Pricing***



In [80]:
from curve_builder import Curve
from hw_model import  HullWhiteCurveBuilder
from hw_pricer import HullWhitePricer
from calibration import HullWhiteCalibrator

import numpy as np
import pandas as pd 
import plotly.express as px 
import QuantLib as ql 

***1. Inputs (EUR)***

In [81]:
# Discount curve data
file = r'Calibration Template Caps EUR.xlsx'
input_curve = pd.read_excel(file)
time = input_curve['Year Frac.']
disc_rate = input_curve['Discount Factor']

# Calibration data
df = pd.read_excel(file, sheet_name = 'Template')
date_columns = df.columns[4:]

market_prices = {
    'Price': df['Price'].tolist(),
    'Strike': df['Strike'].tolist(),
    'Notional': df['Notional'].tolist(),
    'Dates': df[date_columns].apply(lambda row: row.dropna().tolist(), axis=1).tolist()}

***2. Curve construction***

In [82]:
curve = Curve(time, disc_rate)
t = np.linspace(0, 30, 500)
px.line(x = t, y = curve.inst_forward_rate(t), title = "Inst. Forward Curve (EUR)") 

***3. Market calibration (EUR)***

In [83]:
init_params = {'a': 0.01, 'sigma':0.01, 'r0': curve.inst_forward_rate(0)}

# Initialize the curve builder, pricer and calibrator
hw_curve = HullWhiteCurveBuilder(curve, init_params, n_paths=10**6)

# Initialize pricer and calibrator, and calibrate to market
pricer = HullWhitePricer(hw_curve)
calibrator = HullWhiteCalibrator(pricer, market_prices)
result = calibrator.calibrate() 

a: 0.13097, sigma: 0.00756, Error: 1.00321e-01
a: 0.12394, sigma: 0.00820, Error: 4.26664e-02
a: 0.11900, sigma: 0.00883, Error: 2.51386e-02
a: 0.12237, sigma: 0.00883, Error: 2.42047e-02
a: 0.14597, sigma: 0.00902, Error: 1.96710e-02
a: 0.17734, sigma: 0.00941, Error: 1.56726e-02
a: 0.20488, sigma: 0.00983, Error: 1.39354e-02
a: 0.21499, sigma: 0.00995, Error: 1.37842e-02
a: 0.22011, sigma: 0.01000, Error: 1.37516e-02
a: 0.22154, sigma: 0.01001, Error: 1.37474e-02
a: 0.22156, sigma: 0.01001, Error: 1.37473e-02
a: 0.22150, sigma: 0.01001, Error: 1.37473e-02
a: 0.22149, sigma: 0.01001, Error: 1.37473e-02
 
Cap 2y difference:  5.9630%
Cap 3y difference: -8.2334%
Cap 4y difference: -2.8320%
Cap 10y difference:  5.0927%
Cap 20y difference: -0.4135%


***4. Simulation of market-consistent escenarios***

In [84]:
# Update the model with the new parameters
a_opt, sigma_opt = result.x
params = {'a': a_opt, 'sigma':sigma_opt, 'r0': curve.inst_forward_rate(0)}
hw_curve = HullWhiteCurveBuilder(curve, init_params)

# Short-Rate r(t) simulation
r_t = hw_curve.short_rate(t = 1) 
fig = px.histogram(r_t, nbins = 200, title = 'Short-Rate distribution in 1 year')
fig.update_layout(showlegend=False)
fig.show()

In [85]:
# Long Rate R(t, T) simulation
R_t = hw_curve.long_rate(t = 1, T = 20) 
fig = px.histogram(R_t, nbins = 200, title = 'Long-Rate (20 years) distribution in 1 year')
fig.update_layout(showlegend=False)
fig.show()

In [86]:
# Forward Rate F(t, T1, T2) simulation
F_t = hw_curve.forward_rate(t = 1, T1 = 1, T2 = 1.5) 
fig = px.histogram(F_t, nbins = 200, title = 'Forward Rate (6M) distribution in 1 year')
fig.update_layout(showlegend=False)
fig.show()

***5. Pricing vanilla derivatives***

In [87]:
pricer = HullWhitePricer(hw_curve)

# Call option zero cupoun bond
bond_call = pricer.zero_bond_call(T = 1, S = 2, K = 0.9)

# Annual cap
Tau = [0, 1, 2, 3, 4, 5]; N = 1; K = 0.025
cap = pricer.cap(Tau, N, K)
print(f"Cap price: {cap:.6f}, Zero bond call price: {bond_call:.6f}")

Cap price: 0.012852, Zero bond call price: 0.079250


***6. Pricing OTC derivatives***

*EUR 100,000,000 2.04700000% Cap on 3-Month EUR-EURIBOR*

*EUR 100,000,000 2.04700000% Floor on 3-Month EUR-EURIBOR*

*Effective 20-Sep-2026 through 20-Sep-2027*

*MTM:  831,056.60 €*


In [88]:
# Start and end dates 
val_date = ql.Date(31, 3, 2025)
start_date = ql.Date(20, 9, 2026)
end_date   = ql.Date(20, 9, 2027)
day_count = ql.Actual360()

# Payment dates (first not included)
schedule = ql.Schedule(val_date, end_date, ql.Period(ql.Quarterly), ql.TARGET(), ql.Following, ql.Following, ql.DateGeneration.Forward, False)
Tau = [day_count.yearFraction(val_date, d) for d in schedule if d > start_date]

# Strike and notional
K = 0.02047; N = 100_000_000

# Valuation
mtm = 831_056.60
cap = pricer.cap(Tau, N, K)
floor = pricer.floor(Tau, N, K)
straddle = cap + floor
dif = straddle/mtm - 1
print(f"Error:{100*dif: .4f}%")

Error: 0.3568%


***7. Market calibration (USD)***



In [89]:
# Discount curve data
file = r'Calibration Template Caps USD.xlsx'
input_curve = pd.read_excel(file).dropna()
time = input_curve['Year Frac.']
disc_rate = input_curve['Discount Factor']

# Calibration data
df = pd.read_excel(file, sheet_name = 'Template')
date_columns = df.columns[4:]

market_prices = {
    'Price': df['Price'].tolist(),
    'Strike': df['Strike'].tolist(),
    'Notional': df['Notional'].tolist(),
    'Dates': df[date_columns].apply(lambda row: row.dropna().tolist(), axis=1).tolist()}

In [90]:
curve = Curve(time, disc_rate)
t = np.linspace(0, 30, 500)
px.line(x = t, y = curve.inst_forward_rate(t), title = "Inst. Forward Curve (USD)") 

In [91]:
init_params = {'a': 0.01, 'sigma':0.01, 'r0': curve.inst_forward_rate(0)}

# Initialize the curve builder, pricer and calibrator
hw_curve = HullWhiteCurveBuilder(curve, init_params)

# Initialize pricer and calibrator, and calibrate to market
pricer = HullWhitePricer(hw_curve)
calibrator = HullWhiteCalibrator(pricer, market_prices)
result = calibrator.calibrate() 

a: 0.00998, sigma: 0.01093, Error: 1.13083e-02
a: 0.00999, sigma: 0.01097, Error: 1.12529e-02
a: 0.01003, sigma: 0.01098, Error: 1.12238e-02
a: 0.01033, sigma: 0.01101, Error: 1.10666e-02
a: 0.01127, sigma: 0.01108, Error: 1.06526e-02
a: 0.01363, sigma: 0.01121, Error: 9.72035e-03
a: 0.01864, sigma: 0.01139, Error: 7.92345e-03
a: 0.02619, sigma: 0.01158, Error: 5.46988e-03
a: 0.03310, sigma: 0.01163, Error: 3.13695e-03
a: 0.03541, sigma: 0.01161, Error: 2.82666e-03
a: 0.03559, sigma: 0.01161, Error: 2.82219e-03
a: 0.03557, sigma: 0.01161, Error: 2.82216e-03
a: 0.03555, sigma: 0.01161, Error: 2.82216e-03
a: 0.03555, sigma: 0.01161, Error: 2.82216e-03
 
Cap 2y difference:  3.7896%
Cap 3y difference: -2.7660%
Cap 4y difference: -2.3995%
Cap 10y difference:  0.1057%
Cap 20y difference:  0.6644%


***8. Pricing OTC derivatives (USD)***

*Notional Amount: USD 25,000,000.00*

*Effective Date: 03/04/2024*

*Termination Date: 03/04/2029*

*Monthly payments*

*Floor Rate: 3.00%*

*Floating Rate Option: USD-SOFR-OIS Compound 3M*

*MTM: 313,786.47 EUR*

In [92]:
# Start and end dates 
val_date = ql.Date(31, 3, 2025)
start_date = ql.Date(3, 4, 2024)
end_date = ql.Date(3, 4, 2029)
day_count = ql.Actual360()

# Payment dates (first not included)
schedule = ql.Schedule(start_date, end_date, ql.Period(ql.Monthly), ql.TARGET(), ql.Following, ql.Following, ql.DateGeneration.Forward, False)
Tau = [day_count.yearFraction(val_date, d) for d in schedule if d > val_date]
if start_date < val_date: Tau = [0.0] + Tau

# Strike, notional, eurusd rate
K = 0.03; N = 25_000_000; eurusd = 1.0817

# Pricing
mtm =  313_786.47 
floor = pricer.floor(Tau, N, K)/eurusd
dif = floor/mtm - 1
print(f"Error :{100*dif: .4f}%")

Error :-3.3328%


c:\Users\WM692MV\OneDrive - EY\Desktop\Calculadoras\Hull-White-Pricer\hw_pricer.py:110: RuntimeWarning:

divide by zero encountered in scalar divide



***9. Market calibration to swaptions (EUR)***

In [93]:
# Discount curve data
file = r'Calibration Template Swaptions EUR.xlsx'
input_curve = pd.read_excel(file).dropna()
time = input_curve['Year Frac.']
disc_rate = input_curve['Discount Factor']

# Calibration data
df = pd.read_excel(file, sheet_name = 'Template')
date_columns = df.columns[5:]

market_prices = {
    'Price': df['Price'].tolist(),
    'Strike': df['Strike'].tolist(),
    'Notional': df['Notional'].tolist(),
    'Dates': df[date_columns].apply(lambda row: row.dropna().tolist(), axis=1).tolist()}

In [94]:
# Initialize curve and initial parameters
curve = Curve(time, disc_rate)
init_params = {'a': 0.02, 'sigma':0.01, 'r0': curve.inst_forward_rate(0)}

# Initialize the curve builder, pricer and calibrator
hw_curve = HullWhiteCurveBuilder(curve, init_params, n_paths=10**5)

# Initialize pricer and calibrator, and calibrate to market
pricer = HullWhitePricer(hw_curve)
calibrator = HullWhiteCalibrator(pricer, market_prices, calibrate_to='swaptions')
result = calibrator.calibrate() 

a: 0.04514, sigma: 0.00929, Error: 4.69922e-02
a: 0.04212, sigma: 0.00921, Error: 3.98022e-02
a: 0.01881, sigma: 0.00837, Error: 8.91588e-03
a: 0.01624, sigma: 0.00810, Error: 3.85389e-03
a: 0.01615, sigma: 0.00809, Error: 3.85347e-03
a: 0.01616, sigma: 0.00810, Error: 3.85346e-03
a: 0.01616, sigma: 0.00810, Error: 3.85346e-03
 
Swaption 1y x 2y difference:  4.3282%
Swaption 2y x 2y difference: -2.2917%
Swaption 5y x 2y difference: -2.0007%
Swaption 10y x 2y difference: -1.7562%
Swaption 1y x 5y difference:  0.7686%
Swaption 2y x 5y difference: -1.1024%
Swaption 5y x 5y difference: -0.8084%
Swaption 10y x 5y difference:  0.5592%
Swaption 1y x 10y difference:  1.4196%
Swaption 2y x 10y difference: -0.3542%
Swaption 5y x 10y difference: -0.5412%
Swaption 10y x 10y difference:  1.5021%


***10. Pricing OTC derivatives (EUR)***

*Notional Amount: EUR 50,000,000.00*

*Expiry: 19/11/2025*

*Maturity: 19/11/2030*

*Straddle*

*Strike: 2.245%*

*Annual payments*

*MTM: 1,205,303.6 EUR*

In [95]:
# Start and end dates 
val_date = ql.Date(31, 3, 2025)
start_date = ql.Date(19, 11, 2025)
end_date = ql.Date(19, 11, 2030)
day_count = ql.Actual360()

# Payment dates (first not included)
schedule = ql.Schedule(start_date, end_date, ql.Period(ql.Annual), ql.TARGET(), ql.Following, ql.Following, ql.DateGeneration.Forward, False)
Tau = [day_count.yearFraction(val_date, d) for d in schedule if d > val_date]

# Strike, notional
K = 0.02245; N = 50_000_000

# Pricing
mtm =  1_205_303.6
straddle = pricer.swaption(Tau, N, K, payer = True) + pricer.swaption(Tau, N, K, payer = False)
dif = straddle/mtm - 1
print(f"Error :{100*dif: .4f}%")

Error :-1.0585%


***11. Market calibration to swaptions (USD)***

In [96]:
# Discount curve data
file = r'Calibration Template Swaptions OTM USD.xlsx'
input_curve = pd.read_excel(file).dropna()
time = input_curve['Year Frac.']
disc_rate = input_curve['Discount Factor']

# Calibration data
df = pd.read_excel(file, sheet_name = 'Template')
date_columns = df.columns[5:]

market_prices = {
    'Price': df['Price'].tolist(),
    'Strike': df['Strike'].tolist(),
    'Notional': df['Notional'].tolist(),
    'Dates': df[date_columns].apply(lambda row: row.dropna().tolist(), axis=1).tolist()}

In [97]:
# Initialize curve and initial parameters
curve = Curve(time, disc_rate)
init_params = {'a': 0.02, 'sigma':0.01, 'r0': curve.inst_forward_rate(0)}

# Initialize the curve builder, pricer and calibrator
hw_curve = HullWhiteCurveBuilder(curve, init_params)

# Initialize pricer and calibrator, and calibrate to market
pricer = HullWhitePricer(hw_curve)
calibrator = HullWhiteCalibrator(pricer, market_prices, calibrate_to='swaptions')
result = calibrator.calibrate() 

a: 0.01494, sigma: 0.00990, Error: 5.59855e-02
a: 0.01545, sigma: 0.00994, Error: 5.47581e-02
a: 0.02038, sigma: 0.01021, Error: 4.98763e-02
a: 0.02156, sigma: 0.01026, Error: 4.96148e-02
a: 0.02174, sigma: 0.01026, Error: 4.96029e-02
a: 0.02174, sigma: 0.01026, Error: 4.96028e-02
a: 0.02174, sigma: 0.01026, Error: 4.96028e-02
 
Swaption 1y x 2y difference:  1.4123%
Swaption 2y x 2y difference: -9.8321%
Swaption 5y x 2y difference: -5.1765%
Swaption 10y x 2y difference:  0.3571%
Swaption 1y x 5y difference:  8.3707%
Swaption 2y x 5y difference: -5.3504%
Swaption 5y x 5y difference: -4.8478%
Swaption 10y x 5y difference: -0.1625%
Swaption 1y x 10y difference:  13.8253%
Swaption 2y x 10y difference: -4.5306%
Swaption 5y x 10y difference: -6.0198%
Swaption 10y x 10y difference: -0.5604%


***12. Pricing OTC derivatives (USD)***

*Notional Amount: USD 25,000,000.00*

*Expiry: 17/07/2030*

*Maturity: 17/07/2040*

*Straddle*

*Strike: 4.38%*

*Semi-annual payments*

*MTM: 2,684,698 EUR*

In [98]:
# Start and end dates 
val_date = ql.Date(31, 3, 2025)
start_date = ql.Date(17, 7, 2030)
end_date = ql.Date(17, 7, 2040)
day_count = ql.Thirty360(ql.Thirty360.USA)

# Payment dates (first not included)
schedule = ql.Schedule(start_date, end_date, ql.Period(ql.Semiannual), ql.TARGET(), ql.Following, ql.Following, ql.DateGeneration.Forward, False)
Tau = [day_count.yearFraction(val_date, d) for d in schedule if d > val_date]

# Strike, notional
K = 0.0438; N = 25_000_000; eurusd = 1.0817

# Pricing
mtm = 2_684_698 # EUR
straddle = (pricer.swaption(Tau, N, K, payer = True) + pricer.swaption(Tau, N, K, payer = False))/eurusd
dif = straddle/mtm - 1
print(f"Error :{100*dif: .4f}%")

Error :-4.1417%
